In [13]:
import pandas as pd
import openpyxl
import os

In [ ]:
# 1. Configuración de Rutas y Estructuras
CATALOG_PATH = "../DICCIONARIOS/240708 Catalogos.xlsx"
FILES_MADE = [
    "../DATA/Silver/COVID19MEXICO2021/COVID19MEXICO2021.csv",
    "../DATA/Silver/COVID19MEXICO2022/COVID19MEXICO2022.csv",
    "../DATA/Silver/COVID19MEXICO2023/COVID19MEXICO2023.csv",
    "../DATA/Silver/COVID19MEXICO2024/COVID19MEXICO2024.csv"
]

# Definición de Clústeres (DIMENSIONS)
DIMENSIONS = {
    'DIM_Geografico_informacion_paciente': ['ENTIDAD_UM', 'ENTIDAD_NAC'],
    'DIM_Geografico_residencia': ['ENTIDAD_RES', 'MUNICIPIO_RES'],
    'DIM_Geografico_Nacionalidad': ['NACIONALIDAD', 'PAIS_NACIONALIDAD', 'PAIS_ORIGEN'],
    'DIM_Descripcion_del_paciente': ['SEXO', 'EDAD', 'TIPO_PACIENTE'],
    'DIM_Indigena': ['HABLA_LENGUA_INDIG', 'INDIGENA', 'MIGRANTE'],
    'DIM_Comorbilidades_Respiratorias': ['INTUBADO', 'NEUMONIA', 'EPOC', 'ASMA', 'TABAQUISMO'],
    'DIM_Comorbilidades_de_presion': ['DIABETES', 'INMUSUPR', 'HIPERTENSION', 'CARDIOVASCULAR', 'OBESIDAD'],
    'DIM_Otras_caracteristicas_medicas': ['EMBARAZO', 'RENAL_CRONICA', 'OTRA_COM'],
    'DIM_Ubicacion_de_laboratorio': ['ORIGEN', 'SECTOR', 'OTRO_CASO', 'UCI', 'RESULTADO_PCR', 'RESULTADO_PCR_COINFECCION'], # PCR omitido si no existe en data
    'DIM_Antigeno': ['TOMA_MUESTRA_ANTIGENO', 'RESULTADO_ANTIGENO'],
    'DIM_Datos_de_laboratorio': ['TOMA_MUESTRA_LAB', 'RESULTADO_LAB', 'CLASIFICACION_FINAL_COVID', 'CLASIFICACION_FINAL_FLU']
}

# Mapeo de Columna a sheet de Excel
CATALOG_MAP = {
    'ORIGEN': 'ORIGEN', 'SECTOR': 'SECTOR', 'ENTIDAD_UM': 'ENTIDADES',
    'SEXO': 'SEXO', 'ENTIDAD_NAC': 'ENTIDADES', 'ENTIDAD_RES': 'ENTIDADES',
    'MUNICIPIO_RES': 'MUNICIPIOS', 'TIPO_PACIENTE': 'TIPO_PACIENTE',
    'INTUBADO': 'SI_ NO', 'NEUMONIA': 'SI_ NO', 'NACIONALIDAD': 'NACIONALIDAD',
    'EMBARAZO': 'SI_ NO', 'HABLA_LENGUA_INDIG': 'SI_ NO', 'INDIGENA': 'SI_ NO',
    'DIABETES': 'SI_ NO', 'EPOC': 'SI_ NO', 'ASMA': 'SI_ NO', 'INMUSUPR': 'SI_ NO',
    'HIPERTENSION': 'SI_ NO', 'OTRA_COM': 'SI_ NO', 'CARDIOVASCULAR': 'SI_ NO',
    'OBESIDAD': 'SI_ NO', 'RENAL_CRONICA': 'SI_ NO', 'TABAQUISMO': 'SI_ NO',
    'OTRO_CASO': 'SI_ NO', 'TOMA_MUESTRA_LAB': 'SI_ NO', 'RESULTADO_LAB': 'RESULTADO_LAB',
    'RESULTADO_PCR': 'RESULTADO_PCR', 'RESULTADO_PCR_COINFECCION': 'RESULTADO_PCR',
    'TOMA_MUESTRA_ANTIGENO': 'SI_ NO', 'RESULTADO_ANTIGENO': 'RESULTADO_ANTIGENO',
    'CLASIFICACION_FINAL_COVID': 'CLASIFICACION_FINAL_COVID',
    'CLASIFICACION_FINAL_FLU': 'CLASIFICACION_FINAL_FLU', 'MIGRANTE': 'SI_ NO', 'UCI': 'SI_ NO'
}

In [15]:
def catalog_load(ruta_excel):
    """Extrae las traducciones de todas las sheets del Excel dinámicamente."""
    xls = pd.ExcelFile(ruta_excel)
    diccionarys = {}
    for sheet in xls.sheet_names:
        df_sheet = pd.read_excel(xls, sheet_name=sheet)
        # Asume columna 0 = Clave, columna 1 = Descripción
        diccionarys[sheet] = dict(zip(df_sheet.iloc[:, 0], df_sheet.iloc[:, 1]))
    return diccionarys

In [16]:
def unique_convin_extract():
    """Itera sobre los CSV masivos en chunks y extrae combinations empíricas."""
    global_convinations = {dim: [] for dim in DIMENSIONS.keys()}
    
    for file in FILES_MADE:
        if not os.path.exists(file): continue
        print(f"Procesando: {file}...")
        
        lot_iterator = pd.read_csv(file, chunksize=250000, low_memory=False, encoding='latin1')
        for chunk in lot_iterator:
            for name_dim, columns in DIMENSIONS.items():
                # Verificar que las columns existan en el dataset
                columns_presents = [c for c in columns if c in chunk.columns]
                if columns_presents:
                    df_unique = chunk[columns_presents].drop_duplicates()
                    global_convinations[name_dim].append(df_unique)
                    
    return global_convinations

In [ ]:
def process_and_export_dimensions(global_convinations, dict_catalogos):
    """Consolida combinations, aplica traducciones y genera la tabla final."""
    if not os.path.exists("dimensiones"):
        os.makedirs("dimensiones")

    for name_dim, dfs_list in global_convinations.items():
        if not dfs_list: continue
        
        # Consolidación final de la dimensión
        final_dim = pd.concat(dfs_list).drop_duplicates().reset_index(drop=True)
        
        # Mapeo de traducciones
        for col in final_dim.columns:
            if col in CATALOG_MAP:
                name_sheet = CATALOG_MAP[col]
                traduction_dyc = dict_catalogos.get(name_sheet, {})
                final_dim[f'DESC_{col}'] = final_dim[col].map(traduction_dyc)
                # Llenado de nulos/valores texto abiertos
                final_dim[f'DESC_{col}'] = final_dim[f'DESC_{col}'].fillna(final_dim[col].astype(str))

        # Generación de Llave Sustituta
        final_dim.insert(0, f'ID_{name_dim.upper()}', final_dim.index + 1)
        
        # Exportación
        exit_path = f"dimensiones/{name_dim}.csv"
        final_dim.to_csv(exit_path, index=False, encoding='utf-8')
        print(f"Dimensión exportada: {exit_path} | Filas: {len(final_dim)}")

In [18]:

print("1. Cargando catálogos en memoria...")
master_dictionary = catalog_load(CATALOG_PATH)

1. Cargando catálogos en memoria...


In [19]:
print("2. Iniciando extracción en lotes (Chunking)...")
combinations = unique_convin_extract()

2. Iniciando extracción en lotes (Chunking)...
Procesando: ../DATA/Bronce/COVID19MEXICO2021/COVID19MEXICO2021.csv...
Procesando: ../DATA/Bronce/COVID19MEXICO2022/COVID19MEXICO2022.csv...
Procesando: ../DATA/Bronce/COVID19MEXICO2023/COVID19MEXICO2023.csv...
Procesando: ../DATA/Bronce/COVID19MEXICO2024/COVID19MEXICO2024.csv...


In [20]:
print("3. Traduciendo y exportando DIMENSIONS...")
process_and_export_dimensions(combinations, master_dictionary)
print("Pipeline finalizado con éxito.")

3. Traduciendo y exportando DIMENSIONS...
Dimensión exportada: dimensiones_finales/DIM_Geografico_informacion_paciente.csv | Filas: 967
Dimensión exportada: dimensiones_finales/DIM_Geografico_residencia.csv | Filas: 2097
Dimensión exportada: dimensiones_finales/DIM_Geografico_Nacionalidad.csv | Filas: 118
Dimensión exportada: dimensiones_finales/DIM_Descripcion_del_paciente.csv | Filas: 405
Dimensión exportada: dimensiones_finales/DIM_Indigena.csv | Filas: 23
Dimensión exportada: dimensiones_finales/DIM_Comorbilidades_Respiratorias.csv | Filas: 115
Dimensión exportada: dimensiones_finales/DIM_Comorbilidades_de_presion.csv | Filas: 114
Dimensión exportada: dimensiones_finales/DIM_Otras_caracteristicas_medicas.csv | Filas: 33
Dimensión exportada: dimensiones_finales/DIM_Ubicacion_de_laboratorio.csv | Filas: 1255
Dimensión exportada: dimensiones_finales/DIM_Antigeno.csv | Filas: 4
Dimensión exportada: dimensiones_finales/DIM_Datos_de_laboratorio.csv | Filas: 13
Pipeline finalizado con éxi

# USE NEXT CELLS ONLY AFTER RUN "Load_ro_gold" NOTEBOOK

In [5]:
import pyarrow.csv as pv
import pyarrow.parquet as pq
from pathlib import Path

def csv_to_parquet_in_memory(path_csv: str, path_parquet: str):
    table = pv.read_csv(path_csv)
    pq.write_table(table, path_parquet)
    print(f"Conversión exitosa: {path_parquet}")

In [ ]:
csv_to_parquet_in_memory("dimensiones/DIM_Geografico_informacion_paciente.csv", "../DATA/Gold/DIMENSIONES/DIM_Geografico_informacion_paciente.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Comorbilidades_Respiratorias.csv", "../DATA/Gold/DIMENSIONES/DIM_Comorbilidades_Respiratorias.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Datos_de_laboratorio.csv", "../DATA/Gold/DIMENSIONES/DIM_Datos_de_laboratorio.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Otras_caracteristicas_medicas.csv", "../DATA/Gold/DIMENSIONES/DIM_Otras_caracteristicas_medicas.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Antigeno.csv", "../DATA/Gold/DIMENSIONES/DIM_Antigeno.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Geografico_residencia.csv", "../DATA/Gold/DIMENSIONES/DIM_Geografico_residencia.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Ubicacion_de_laboratorio.csv", "../DATA/Gold/DIMENSIONES/DIM_Ubicacion_de_laboratorio.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Comorbilidades_de_presion.csv", "../DATA/Gold/DIMENSIONES/DIM_Comorbilidades_de_presion.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Indigena.csv", "../DATA/Gold/DIMENSIONES/DIM_Indigena.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Descripcion_del_paciente.csv", "../DATA/Gold/DIMENSIONES/DIM_Descripcion_del_paciente.parquet")
csv_to_parquet_in_memory("dimensiones/DIM_Geografico_Nacionalidad.csv", "../DATA/Gold/DIMENSIONES/DIM_Geografico_Nacionalidad.parquet")

Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Comorbilidades_Respiratorias.parquet
Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Datos_de_laboratorio.parquet
Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Otras_caracteristicas_medicas.parquet
Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Antigeno.parquet
Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Geografico_residencia.parquet
Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Ubicacion_de_laboratorio.parquet
Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Comorbilidades_de_presion.parquet
Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Indigena.parquet
Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Descripcion_del_paciente.parquet
Conversión exitosa: ../DATA/Gold/DIMENSIONES/DIM_Geografico_Nacionalidad.parquet
